# E-Commerce Sales Analysis

This notebook analyzes e-commerce transaction data to evaluate business performance across four dimensions:
revenue trends, product category performance, geographic distribution, and customer experience.

The analysis compares a configurable primary period against a prior-year baseline.
All parameters are set in the **Configuration** section below.

## Table of Contents

1. [Introduction and Business Objectives](#1-introduction-and-business-objectives)
2. [Data Dictionary](#2-data-dictionary)
3. [Configuration](#3-configuration)
4. [Data Loading](#4-data-loading)
5. [Data Preparation and Transformation](#5-data-preparation-and-transformation)
6. [Business Metrics](#6-business-metrics)
   - 6.1 [Revenue Analysis](#61-revenue-analysis)
   - 6.2 [Product Category Analysis](#62-product-category-analysis)
   - 6.3 [Geographic Analysis](#63-geographic-analysis)
   - 6.4 [Customer Experience Analysis](#64-customer-experience-analysis)
7. [Summary of Observations](#7-summary-of-observations)

## 1. Introduction and Business Objectives

The dataset covers e-commerce orders placed on a US marketplace. Six tables are available:
orders, order items, products, customers, reviews, and payments.

Key questions this analysis addresses:

- How did total revenue and order volume change year-over-year?
- What is the month-over-month revenue growth trend within the analysis period?
- Which product categories generate the most revenue?
- Which US states produce the highest sales volume?
- How does delivery speed correlate with customer review scores?
- What share of orders reach delivered status versus other outcomes?

## 2. Data Dictionary

### Orders (`orders_dataset.csv`)

| Column | Description |
|--------|-------------|
| order_id | Unique order identifier |
| customer_id | Customer identifier (links to customers table) |
| order_status | Order lifecycle state: delivered, canceled, shipped, etc. |
| order_purchase_timestamp | Date and time the order was placed |
| order_approved_at | Date and time payment was approved |
| order_delivered_carrier_date | Date the order was handed to the carrier |
| order_delivered_customer_date | Date the order was delivered to the customer |
| order_estimated_delivery_date | Seller-estimated delivery date |

### Order Items (`order_items_dataset.csv`)

| Column | Description |
|--------|-------------|
| order_id | Order identifier (foreign key) |
| order_item_id | Item sequence number within the order |
| product_id | Product identifier (foreign key) |
| seller_id | Seller identifier |
| price | Item price in USD |
| freight_value | Shipping cost in USD |

### Products (`products_dataset.csv`)

| Column | Description |
|--------|-------------|
| product_id | Unique product identifier |
| product_category_name | Category the product belongs to |

### Customers (`customers_dataset.csv`)

| Column | Description |
|--------|-------------|
| customer_id | Order-level customer identifier |
| customer_unique_id | Unique customer identifier across multiple orders |
| customer_state | Two-letter US state abbreviation |
| customer_city | Customer city name |

### Order Reviews (`order_reviews_dataset.csv`)

| Column | Description |
|--------|-------------|
| review_id | Unique review identifier |
| order_id | Order being reviewed (foreign key) |
| review_score | Rating from 1 (lowest) to 5 (highest) |
| review_comment_title | Short review title |
| review_comment_message | Full review text |

### Business Terms

| Term | Definition |
|------|------------|
| Revenue | Sum of item prices for delivered orders |
| Average Order Value (AOV) | Mean total price per unique order |
| YoY Growth | Year-over-year percentage change |
| MoM Growth | Month-over-month percentage change |
| Delivery Days | Days from order purchase to customer delivery |

## 3. Configuration

Set the analysis period below. Re-run all cells after changing any value.

- `ANALYSIS_MONTH = None` analyzes the full year.
- Set `ANALYSIS_MONTH` to an integer (1-12) to restrict to a single calendar month.

In [ ]:
# Analysis period
ANALYSIS_YEAR   = 2023   # Primary year to analyze
COMPARISON_YEAR = 2022   # Baseline year for year-over-year comparisons
ANALYSIS_MONTH  = None   # Integer 1-12 for a single month, or None for a full year

DATA_DIR = 'ecommerce_data'

## 4. Data Loading

All six datasets are loaded from the configured data directory.
Row counts are printed for a quick sanity check.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px

from data_loader import load_data, build_sales_data, filter_by_period
from business_metrics import (
    calculate_revenue,
    calculate_revenue_growth,
    calculate_monthly_revenue,
    calculate_monthly_growth,
    calculate_avg_order_value,
    calculate_total_orders,
    calculate_product_sales,
    calculate_sales_by_state,
    calculate_delivery_metrics,
    calculate_avg_review_by_delivery,
    calculate_review_distribution,
    calculate_order_status_distribution,
)

orders, order_items, products, customers, reviews, order_payments = load_data(DATA_DIR)

print(f'Orders:       {len(orders):>8,} rows')
print(f'Order items:  {len(order_items):>8,} rows')
print(f'Products:     {len(products):>8,} rows')
print(f'Customers:    {len(customers):>8,} rows')
print(f'Reviews:      {len(reviews):>8,} rows')
print(f'Payments:     {len(order_payments):>8,} rows')

## 5. Data Preparation and Transformation

Order items are joined with orders and filtered to delivered orders only.
Timestamps are parsed and year/month columns are added to support period filtering.
The resulting `sales_delivered` DataFrame is then split into the analysis and comparison periods.

In [ ]:
# Build unified sales dataset (delivered orders only)
sales_delivered = build_sales_data(orders, order_items)

# Filter to the configured periods
sales_current = filter_by_period(sales_delivered, ANALYSIS_YEAR, ANALYSIS_MONTH)
sales_prior   = filter_by_period(sales_delivered, COMPARISON_YEAR, ANALYSIS_MONTH)

# Human-readable labels used throughout the notebook
if ANALYSIS_MONTH:
    period_label = f'{ANALYSIS_YEAR}-{ANALYSIS_MONTH:02d}'
    prior_label  = f'{COMPARISON_YEAR}-{ANALYSIS_MONTH:02d}'
else:
    period_label = str(ANALYSIS_YEAR)
    prior_label  = str(COMPARISON_YEAR)

print(f'Analysis period  : {period_label}  '
      f'({sales_current["order_id"].nunique():,} orders, '
      f'{len(sales_current):,} line items)')
print(f'Comparison period: {prior_label}  '
      f'({sales_prior["order_id"].nunique():,} orders, '
      f'{len(sales_prior):,} line items)')

## 6. Business Metrics

### 6.1 Revenue Analysis

Total revenue, order volume, and average order value for the analysis period,
each compared against the prior-year baseline.

In [ ]:
revenue_current = calculate_revenue(sales_current)
revenue_prior   = calculate_revenue(sales_prior)
revenue_growth  = calculate_revenue_growth(revenue_current, revenue_prior)

orders_current = calculate_total_orders(sales_current)
orders_prior   = calculate_total_orders(sales_prior)
orders_growth  = calculate_revenue_growth(orders_current, orders_prior)

aov_current = calculate_avg_order_value(sales_current)
aov_prior   = calculate_avg_order_value(sales_prior)
aov_growth  = calculate_revenue_growth(aov_current, aov_prior)

def fmt_growth(g):
    return f'{g * 100:+.1f}%' if g is not None else 'N/A'

print(f'Revenue ({period_label}):          ${revenue_current:>12,.2f}')
print(f'Revenue ({prior_label}):          ${revenue_prior:>12,.2f}')
print(f'Revenue YoY growth:               {fmt_growth(revenue_growth):>8}')
print()
print(f'Total orders ({period_label}):    {orders_current:>12,}')
print(f'Total orders ({prior_label}):    {orders_prior:>12,}')
print(f'Orders YoY growth:                {fmt_growth(orders_growth):>8}')
print()
print(f'Avg order value ({period_label}):  ${aov_current:>11,.2f}')
print(f'Avg order value ({prior_label}):  ${aov_prior:>11,.2f}')
print(f'AOV YoY growth:                   {fmt_growth(aov_growth):>8}')

#### Monthly Revenue Trend

Total revenue per calendar month for the analysis period.

In [ ]:
PALETTE = {
    'primary':   '#2C5F8A',
    'secondary': '#4A90D9',
    'positive':  '#2C7A4B',
    'negative':  '#C0392B',
    'light':     '#A8C8E8',
    'grid':      '#E8EDF2',
}

monthly_rev = calculate_monthly_revenue(sales_current)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_rev.index, monthly_rev.values, color=PALETTE['primary'], width=0.6)
ax.set_title(f'Monthly Revenue ({period_label})', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Revenue (USD)', fontsize=11)
ax.set_xticks(monthly_rev.index)
ax.set_xticklabels(
    [pd.Timestamp(2020, m, 1).strftime('%b') for m in monthly_rev.index]
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_facecolor(PALETTE['grid'])
ax.grid(axis='y', color='white', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

#### Month-over-Month Revenue Growth

Percentage change in revenue from one month to the next within the analysis period.
This chart is only shown when a full year is selected (ANALYSIS_MONTH = None).

In [ ]:
if ANALYSIS_MONTH is None:
    monthly_growth = calculate_monthly_growth(sales_current).dropna()

    colors = [
        PALETTE['positive'] if v >= 0 else PALETTE['negative']
        for v in monthly_growth.values
    ]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(monthly_growth.index, monthly_growth.values * 100, color=colors, width=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(
        f'Month-over-Month Revenue Growth ({period_label})',
        fontsize=14, fontweight='bold', pad=12
    )
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel('Growth (%)', fontsize=11)
    ax.set_xticks(monthly_growth.index)
    ax.set_xticklabels(
        [pd.Timestamp(2020, m, 1).strftime('%b') for m in monthly_growth.index]
    )
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:+.1f}%')
    )
    ax.set_facecolor(PALETTE['grid'])
    ax.grid(axis='y', color='white', linewidth=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

    mean_growth = monthly_growth.mean()
    print(f'Mean month-over-month growth ({period_label}): {mean_growth * 100:+.1f}%')
else:
    print('Month-over-month growth requires a full-year selection (set ANALYSIS_MONTH = None).')

### 6.2 Product Category Analysis

Revenue breakdown by product category for the analysis period,
highlighting which categories drive the most value.

In [ ]:
product_sales = calculate_product_sales(sales_current, products)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(product_sales.index, product_sales.values, color=PALETTE['secondary'], width=0.7)
ax.set_title(
    f'Revenue by Product Category ({period_label})',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Product Category', fontsize=11)
ax.set_ylabel('Revenue (USD)', fontsize=11)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_facecolor(PALETTE['grid'])
ax.grid(axis='y', color='white', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

total_product_rev = product_sales.sum()
print(f'\nTop 5 categories by revenue ({period_label}):')
for cat, rev in product_sales.head(5).items():
    share = rev / total_product_rev * 100
    print(f'  {cat:<35} ${rev:>10,.2f}  ({share:.1f}%)')

### 6.3 Geographic Analysis

Revenue distribution across US states, shown as a choropleth map and a ranked table.

In [ ]:
sales_by_state = calculate_sales_by_state(sales_current, orders, customers)

fig = px.choropleth(
    sales_by_state,
    locations='customer_state',
    color='price',
    locationmode='USA-states',
    scope='usa',
    title=f'Revenue by State ({period_label})',
    color_continuous_scale='Blues',
    labels={'price': 'Revenue (USD)', 'customer_state': 'State'},
)
fig.update_layout(
    title_font_size=16,
    coloraxis_colorbar=dict(title='Revenue (USD)', tickformat='$,.0f'),
)
fig.show()

total_state_rev = sales_by_state['price'].sum()
print(f'\nTop 10 states by revenue ({period_label}):')
for _, row in sales_by_state.head(10).iterrows():
    share = row['price'] / total_state_rev * 100
    print(f'  {row["customer_state"]:<6} ${row["price"]:>12,.2f}  ({share:.1f}%)')

### 6.4 Customer Experience Analysis

Delivery performance and customer satisfaction metrics.
Delivery speed is grouped into three buckets: 1-3 days, 4-7 days, and 8+ days.

In [ ]:
delivery_df = calculate_delivery_metrics(sales_current, reviews)

avg_delivery = delivery_df['delivery_days'].mean()
avg_review   = delivery_df['review_score'].mean()

print(f'Average delivery time ({period_label}): {avg_delivery:.1f} days')
print(f'Average review score  ({period_label}): {avg_review:.2f} / 5.0')

#### Review Score Distribution

Proportion of orders receiving each review score (1 = lowest, 5 = highest).

In [ ]:
review_dist = calculate_review_distribution(delivery_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(
    review_dist.index.astype(int).astype(str),
    review_dist.values * 100,
    color=PALETTE['primary'],
    height=0.55,
)
ax.set_title(
    f'Review Score Distribution ({period_label})',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Share of Orders (%)', fontsize=11)
ax.set_ylabel('Review Score', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.set_facecolor(PALETTE['grid'])
ax.grid(axis='x', color='white', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

#### Average Review Score by Delivery Speed

Mean customer rating for each delivery speed bucket.

In [ ]:
bucket_order = ['1-3 days', '4-7 days', '8+ days']
review_by_delivery = calculate_avg_review_by_delivery(delivery_df)
review_by_delivery['delivery_bucket'] = pd.Categorical(
    review_by_delivery['delivery_bucket'], categories=bucket_order, ordered=True
)
review_by_delivery = review_by_delivery.sort_values('delivery_bucket')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    review_by_delivery['delivery_bucket'],
    review_by_delivery['review_score'],
    color=PALETTE['secondary'],
    width=0.45,
)
ax.set_title(
    f'Average Review Score by Delivery Speed ({period_label})',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Delivery Time', fontsize=11)
ax.set_ylabel('Average Review Score', fontsize=11)
ax.set_ylim(0, 5)
ax.set_facecolor(PALETTE['grid'])
ax.grid(axis='y', color='white', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nAverage review score by delivery bucket ({period_label}):')
for _, row in review_by_delivery.iterrows():
    print(f'  {row["delivery_bucket"]:<12} {row["review_score"]:.2f} / 5.0')

#### Order Status Distribution

Share of all orders by status for the analysis year (includes non-delivered orders).

In [ ]:
status_dist = calculate_order_status_distribution(orders, ANALYSIS_YEAR)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(
    status_dist.index,
    status_dist.values * 100,
    color=PALETTE['light'],
    edgecolor=PALETTE['primary'],
    height=0.55,
)
ax.set_title(
    f'Order Status Distribution ({ANALYSIS_YEAR})',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Share of Orders (%)', fontsize=11)
ax.set_ylabel('Order Status', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.set_facecolor(PALETTE['grid'])
ax.grid(axis='x', color='white', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nOrder status breakdown ({ANALYSIS_YEAR}):')
for status, share in status_dist.items():
    print(f'  {status:<20} {share * 100:.1f}%')

## 7. Summary of Observations

The sections above cover the following dimensions for the configured period:

**Revenue**
- Total revenue and year-over-year change relative to the prior period.
- Monthly revenue trend and mean month-over-month growth rate.
- Average order value and its year-over-year change.

**Products**
- Revenue by product category, identifying the highest-contributing segments.

**Geography**
- Revenue distribution across US states via choropleth map and ranked table.

**Customer Experience**
- Average delivery time in days.
- Review score distribution (1-5 scale).
- Correlation between delivery speed buckets and average review score.
- Order status breakdown for the analysis year.

To run this analysis for a different period, update `ANALYSIS_YEAR`, `COMPARISON_YEAR`,
and `ANALYSIS_MONTH` in Section 3 and run **Kernel > Restart and Run All**.